# What's In My Thattu — Food Recognition Model Training

This notebook trains a food recognition model using **EfficientNetV2B0** transfer learning on the **Food-101** dataset, then exports it as a `.tflite` file ready for the Android app.

## Setup
- **Runtime**: Kaggle GPU (T4) — enable via *Settings → Accelerator → GPU T4 x2*
- **Dataset**: Food-101 loaded via `tensorflow_datasets` (auto-downloaded)
- **Output**: `whats_in_my_thattu_v2.tflite` in `/kaggle/working/` (download from Output tab)

## Training Strategy
1. **Phase 1** — Transfer learning with frozen base (higher LR, fast convergence)
2. **Phase 2** — Fine-tuning top layers (low LR, refines food-specific features)

## 1. Environment Check & Installs

In [ ]:
import subprocess
import sys

# Install only what Kaggle doesn't already have
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "tflite-support", "tensorflow-datasets"])

import tensorflow as tf
import numpy as np
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")

# Configure GPU memory growth to avoid OOM on Kaggle free tier
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

if not gpus:
    print("WARNING: No GPU detected! Enable GPU in Settings → Accelerator → GPU T4 x2")

## 2. Configuration

All hyperparameters in one place. Tuned for Kaggle free-tier T4 GPU (~16 GB VRAM).

In [ ]:
# === Paths (Kaggle-specific) ===
OUTPUT_DIR = "/kaggle/working"
TFLITE_MODEL_PATH = os.path.join(OUTPUT_DIR, "whats_in_my_thattu_v2.tflite")
LABEL_MAP_PATH = os.path.join(OUTPUT_DIR, "labels.txt")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# === Dataset ===
DATASET_NAME = "food101"
NUM_CLASSES = 101
VALIDATION_SPLIT = 0.15

# === Image ===
IMAGE_SIZE = 224
CHANNELS = 3

# === Training — Phase 1 (Transfer Learning, frozen base) ===
BATCH_SIZE = 32
PHASE1_EPOCHS = 20
PHASE1_LR = 1e-3

# === Training — Phase 2 (Fine-tuning, unfrozen top layers) ===
PHASE2_EPOCHS = 15
PHASE2_LR = 1e-5
FINE_TUNE_AT_LAYER = 100  # Unfreeze from this layer onwards

# === Regularization ===
DROPOUT_RATE = 0.3
LABEL_SMOOTHING = 0.1
WEIGHT_DECAY = 1e-5

# === Callbacks ===
EARLY_STOPPING_PATIENCE = 5
REDUCE_LR_PATIENCE = 3
REDUCE_LR_FACTOR = 0.5
MIN_LR = 1e-7

print("Configuration loaded.")
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Phase 1: {PHASE1_EPOCHS} epochs @ LR={PHASE1_LR}")
print(f"  Phase 2: {PHASE2_EPOCHS} epochs @ LR={PHASE2_LR}")

## 3. Load & Prepare the Food-101 Dataset

Food-101 contains **101,000 images** across 101 food categories.
We apply data augmentation only to training data to reduce overfitting.

In [ ]:
import tensorflow_datasets as tfds

# --- Preprocessing functions ---

def decode_and_resize(image, label):
    """Resize image to model input size."""
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    return image, label

def normalize(image, label):
    """Normalize pixels to [0, 1]."""
    image = image / 255.0
    return image, label

def augment(image, label):
    """Apply data augmentation to training images."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    
    # Random crop and resize back (simulates zoom)
    crop_size = tf.random.uniform([], minval=int(IMAGE_SIZE * 0.85), maxval=IMAGE_SIZE, dtype=tf.int32)
    image = tf.image.random_crop(image, size=[crop_size, crop_size, CHANNELS])
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

def one_hot(image, label):
    """One-hot encode the label."""
    return image, tf.one_hot(label, NUM_CLASSES)

# --- Load dataset ---

print("Loading Food-101 dataset (this may take a few minutes on first run)...")

# Get class names first
builder = tfds.builder(DATASET_NAME)
info = builder.info
class_names = info.features["label"].names
print(f"Classes: {NUM_CLASSES}")
print(f"Sample classes: {class_names[:10]}")

# Save label map for TFLite metadata
with open(LABEL_MAP_PATH, "w") as f:
    for name in class_names:
        display_name = name.replace("_", " ").title()
        f.write(f"{display_name}\n")
print(f"Label map saved: {LABEL_MAP_PATH}")

# Split: 85% train, 15% val from the training set; use 'validation' as test
val_pct = int(VALIDATION_SPLIT * 100)
train_pct = 100 - val_pct

train_ds, val_ds = tfds.load(
    DATASET_NAME,
    split=[f"train[:{train_pct}%]", f"train[{train_pct}%:]"],
    as_supervised=True,
)

test_ds = tfds.load(
    DATASET_NAME,
    split="validation",  # Food-101's 'validation' split is the test set
    as_supervised=True,
)

# --- Build pipelines ---

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    train_ds
    .shuffle(10000)
    .map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .map(normalize, num_parallel_calls=AUTOTUNE)
    .map(one_hot, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds
    .map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    .map(normalize, num_parallel_calls=AUTOTUNE)
    .map(one_hot, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    test_ds
    .map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    .map(normalize, num_parallel_calls=AUTOTUNE)
    .map(one_hot, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f"\nPipeline ready:")
print(f"  Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"  Val batches:   {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"  Test batches:  {tf.data.experimental.cardinality(test_ds).numpy()}")

## 4. Visualize Sample Training Images

Quick sanity check to verify augmentation and labels look correct.

In [ ]:
import matplotlib.pyplot as plt

# Get one batch
sample_images, sample_labels = next(iter(train_ds))

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, ax in enumerate(axes.flat):
    if i < len(sample_images):
        ax.imshow(sample_images[i].numpy())
        label_idx = tf.argmax(sample_labels[i]).numpy()
        ax.set_title(class_names[label_idx].replace("_", " ").title(), fontsize=10)
    ax.axis("off")

plt.suptitle("Sample Training Images (with augmentation)", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Build the Model

**Architecture**: EfficientNetV2B0 (pretrained on ImageNet) + custom classification head.

EfficientNetV2B0 was chosen because:
- Excellent accuracy-to-size ratio (best for mobile deployment)
- 78.7% ImageNet top-1 (vs ~70% for the old AIY model's MobileNet base)
- ~7 MB base model size → compact TFLite output

In [ ]:
from tensorflow.keras import layers, regularizers

def build_model(num_classes, fine_tune=False):
    """Build food classifier with EfficientNetV2B0 backbone."""
    
    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
    
    # Base model
    base_model = tf.keras.applications.EfficientNetV2B0(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs,
        include_preprocessing=False,  # We normalize in the data pipeline
    )
    
    if fine_tune:
        base_model.trainable = True
        for layer in base_model.layers[:FINE_TUNE_AT_LAYER]:
            layer.trainable = False
        trainable = sum(1 for l in base_model.layers if l.trainable)
        print(f"Fine-tune mode: {trainable} layers unfrozen")
    else:
        base_model.trainable = False
        print("Transfer learning mode: base model frozen")
    
    # Classification head
    x = base_model.output
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.BatchNormalization(name="bn_1")(x)
    x = layers.Dense(512, activation="relu",
                     kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                     name="dense_1")(x)
    x = layers.Dropout(DROPOUT_RATE, name="dropout_1")(x)
    x = layers.BatchNormalization(name="bn_2")(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                     name="dense_2")(x)
    x = layers.Dropout(DROPOUT_RATE, name="dropout_2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="food_classifier_v2")
    return model, base_model

# Build with frozen base
model, base_model = build_model(NUM_CLASSES, fine_tune=False)
model.summary(show_trainable=True)

## 6. Phase 1 — Transfer Learning (Frozen Base)

Train only the classification head with the EfficientNet base frozen.
This lets the head learn food-specific features quickly without disturbing the pretrained weights.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE1_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top5_accuracy"),
    ],
)

phase1_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, "best_phase1.keras"),
        monitor="val_accuracy", mode="max",
        save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=EARLY_STOPPING_PATIENCE,
        mode="max", restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=1,
    ),
]

print("=" * 60)
print("PHASE 1: Transfer Learning (frozen base)")
print("=" * 60)

history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=phase1_callbacks,
)

print(f"\nPhase 1 complete!")
print(f"  Best val accuracy: {max(history_p1.history['val_accuracy'])*100:.2f}%")
print(f"  Best val top-5:    {max(history_p1.history['val_top5_accuracy'])*100:.2f}%")

## 7. Phase 2 — Fine-Tuning (Unfreeze Top Layers)

Unfreeze the top layers of EfficientNetV2B0 and train end-to-end with a much lower learning rate.
This lets the pretrained features adapt to food-specific patterns.

In [ ]:
# Unfreeze top layers of the base model
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT_LAYER]:
    layer.trainable = False

unfrozen = sum(1 for l in base_model.layers if l.trainable)
frozen = sum(1 for l in base_model.layers if not l.trainable)
print(f"Fine-tuning: {unfrozen} unfrozen, {frozen} frozen layers")

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE2_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top5_accuracy"),
    ],
)

phase2_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, "best_phase2.keras"),
        monitor="val_accuracy", mode="max",
        save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=EARLY_STOPPING_PATIENCE,
        mode="max", restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=1,
    ),
]

print("\n" + "=" * 60)
print("PHASE 2: Fine-Tuning (top layers unfrozen)")
print("=" * 60)

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=phase2_callbacks,
)

print(f"\nPhase 2 complete!")
print(f"  Best val accuracy: {max(history_p2.history['val_accuracy'])*100:.2f}%")
print(f"  Best val top-5:    {max(history_p2.history['val_top5_accuracy'])*100:.2f}%")

## 8. Training History Plots

In [ ]:
import matplotlib.pyplot as plt

def plot_history(h1, h2):
    """Plot training curves for both phases."""
    # Combine histories
    acc = h1.history["accuracy"] + h2.history["accuracy"]
    val_acc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
    loss = h1.history["loss"] + h2.history["loss"]
    val_loss = h1.history["val_loss"] + h2.history["val_loss"]
    top5 = h1.history["top5_accuracy"] + h2.history["top5_accuracy"]
    val_top5 = h1.history["val_top5_accuracy"] + h2.history["val_top5_accuracy"]
    
    epochs = range(1, len(acc) + 1)
    phase1_end = len(h1.history["accuracy"])
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Accuracy
    axes[0].plot(epochs, acc, "b-", label="Train")
    axes[0].plot(epochs, val_acc, "r-", label="Validation")
    axes[0].axvline(x=phase1_end, color="gray", linestyle="--", alpha=0.7, label="Fine-tune start")
    axes[0].set_title("Top-1 Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Top-5 Accuracy
    axes[1].plot(epochs, top5, "b-", label="Train")
    axes[1].plot(epochs, val_top5, "r-", label="Validation")
    axes[1].axvline(x=phase1_end, color="gray", linestyle="--", alpha=0.7, label="Fine-tune start")
    axes[1].set_title("Top-5 Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Loss
    axes[2].plot(epochs, loss, "b-", label="Train")
    axes[2].plot(epochs, val_loss, "r-", label="Validation")
    axes[2].axvline(x=phase1_end, color="gray", linestyle="--", alpha=0.7, label="Fine-tune start")
    axes[2].set_title("Loss")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Loss")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle("Training History (Phase 1 → Phase 2)", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()

plot_history(history_p1, history_p2)

## 9. Evaluate on Test Set

In [ ]:
print("Evaluating on test set (25,250 images)...")
test_results = model.evaluate(test_ds, verbose=1)

print(f"\n{'='*50}")
print(f"TEST RESULTS")
print(f"{'='*50}")
print(f"  Loss:          {test_results[0]:.4f}")
print(f"  Top-1 Accuracy: {test_results[1]*100:.2f}%")
print(f"  Top-5 Accuracy: {test_results[2]*100:.2f}%")

## 10. Per-Class Analysis

See which food categories the model is best/worst at recognizing.

In [ ]:
from collections import Counter

# Collect all predictions
all_preds = []
all_labels = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    all_preds.append(preds)
    all_labels.append(labels.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

pred_classes = np.argmax(all_preds, axis=1)
true_classes = np.argmax(all_labels, axis=1)

# Per-class accuracy
per_class = []
for i, name in enumerate(class_names):
    mask = true_classes == i
    if mask.sum() == 0:
        continue
    acc = np.mean(pred_classes[mask] == i)
    per_class.append((name.replace("_", " ").title(), acc, mask.sum()))

per_class.sort(key=lambda x: x[1])

print("WORST 10 classes:")
print("-" * 50)
for name, acc, count in per_class[:10]:
    print(f"  {name:30s} {acc*100:5.1f}% ({count} samples)")

print(f"\nBEST 10 classes:")
print("-" * 50)
for name, acc, count in per_class[-10:]:
    print(f"  {name:30s} {acc*100:5.1f}% ({count} samples)")

# Most confused pairs
confusion_pairs = Counter()
for t, p in zip(true_classes, pred_classes):
    if t != p:
        confusion_pairs[(class_names[t], class_names[p])] += 1

print(f"\nMOST CONFUSED PAIRS:")
print("-" * 60)
for (true_name, pred_name), count in confusion_pairs.most_common(10):
    t = true_name.replace("_", " ").title()
    p = pred_name.replace("_", " ").title()
    print(f"  {t:25s} → {p:25s} ({count} times)")

## 11. Export to TFLite

Convert the trained model to TensorFlow Lite format with **float16 quantization**.
This reduces model size by ~50% with negligible accuracy loss — ideal for mobile.

The exported `.tflite` file will appear in the **Output** tab (right sidebar) for download.

In [ ]:
# --- Convert to TFLite with float16 quantization ---

print("Converting model to TFLite (float16 quantization)...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

# Save
with open(TFLITE_MODEL_PATH, "wb") as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_MODEL_PATH) / (1024 * 1024)
print(f"\nTFLite model saved: {TFLITE_MODEL_PATH}")
print(f"Model size: {size_mb:.1f} MB")

In [ ]:
# --- Add metadata (labels) to the TFLite model ---

try:
    from tflite_support.metadata_writers import image_classifier
    from tflite_support.metadata_writers import writer_utils

    writer = image_classifier.MetadataWriter.create_for_inference(
        writer_utils.load_file(TFLITE_MODEL_PATH),
        model_name="What's In My Thattu Food Classifier",
        model_description=(
            "Food recognition model trained on Food-101 using EfficientNetV2B0. "
            "Classifies images into 101 food categories."
        ),
        input_norm_mean=[0.0],
        input_norm_std=[255.0],
        label_file_paths=[LABEL_MAP_PATH],
    )

    writer_utils.save_file(writer.populate(), TFLITE_MODEL_PATH)
    print("Metadata with labels embedded into TFLite model.")

except Exception as e:
    print(f"Metadata embedding skipped ({e}).")
    print("The model still works — labels.txt is saved separately as fallback.")
    # Copy labels alongside model as fallback
    import shutil
    fallback_labels = TFLITE_MODEL_PATH.replace(".tflite", "_labels.txt")
    shutil.copy(LABEL_MAP_PATH, fallback_labels)
    print(f"Fallback labels saved: {fallback_labels}")

## 12. Verify TFLite Model

In [ ]:
# --- Verify the exported model works ---

print("Verifying TFLite model...")

interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"  Input:  shape={input_details[0]['shape']}, dtype={input_details[0]['dtype']}")
print(f"  Output: shape={output_details[0]['shape']}, dtype={output_details[0]['dtype']}")

# Test with a real image from the test set
test_batch = next(iter(test_ds))
test_image = test_batch[0][0:1].numpy().astype(np.float32)
true_label = np.argmax(test_batch[1][0].numpy())

interpreter.set_tensor(input_details[0]["index"], test_image)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]["index"])[0]

top5_indices = np.argsort(output)[::-1][:5]
true_name = class_names[true_label].replace("_", " ").title()

print(f"\n  True label: {true_name}")
print(f"  TFLite Top-5 predictions:")
for i, idx in enumerate(top5_indices, 1):
    name = class_names[idx].replace("_", " ").title()
    print(f"    {i}. {name:25s} {output[idx]*100:.1f}%")

print(f"\n  Softmax sum: {output.sum():.4f} (should be ~1.0)")
print(f"  Verification: PASSED")

## 13. TFLite Accuracy Check

Run the TFLite model on a subset of test images to verify accuracy is preserved after quantization.

In [ ]:
# Quick accuracy check on 1000 test images
NUM_EVAL_SAMPLES = 1000

correct = 0
correct_top5 = 0
total = 0

for images, labels in test_ds:
    for i in range(images.shape[0]):
        if total >= NUM_EVAL_SAMPLES:
            break
        
        img = images[i:i+1].numpy().astype(np.float32)
        true_label = np.argmax(labels[i].numpy())
        
        interpreter.set_tensor(input_details[0]["index"], img)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])[0]
        
        pred = np.argmax(output)
        top5 = np.argsort(output)[::-1][:5]
        
        if pred == true_label:
            correct += 1
        if true_label in top5:
            correct_top5 += 1
        total += 1
    
    if total >= NUM_EVAL_SAMPLES:
        break

print(f"TFLite accuracy on {total} test samples:")
print(f"  Top-1: {correct/total*100:.2f}%")
print(f"  Top-5: {correct_top5/total*100:.2f}%")

## 14. Save Final Keras Model & Summary

In [ ]:
# Save Keras model too (in case you want to resume training later)
keras_path = os.path.join(OUTPUT_DIR, "food_classifier_v2.keras")
model.save(keras_path)
keras_size = os.path.getsize(keras_path) / (1024 * 1024)

tflite_size = os.path.getsize(TFLITE_MODEL_PATH) / (1024 * 1024)

print("\n" + "=" * 60)
print("TRAINING COMPLETE — SUMMARY")
print("=" * 60)
print(f"")
print(f"  Phase 1 best val accuracy:  {max(history_p1.history['val_accuracy'])*100:.2f}%")
print(f"  Phase 2 best val accuracy:  {max(history_p2.history['val_accuracy'])*100:.2f}%")
print(f"  Test top-1 accuracy:        {test_results[1]*100:.2f}%")
print(f"  Test top-5 accuracy:        {test_results[2]*100:.2f}%")
print(f"")
print(f"  Keras model:  {keras_path} ({keras_size:.1f} MB)")
print(f"  TFLite model: {TFLITE_MODEL_PATH} ({tflite_size:.1f} MB)")
print(f"  Labels:       {LABEL_MAP_PATH}")
print(f"")
print(f"  OUTPUT FILES (download from the Output tab on the right):")
print(f"    - whats_in_my_thattu_v2.tflite  ← Deploy this to Android")
print(f"    - labels.txt")
print(f"    - food_classifier_v2.keras")
print(f"    - training_curves.png")

## 15. Deploy to Android

After downloading `whats_in_my_thattu_v2.tflite` from the **Output** tab:

```bash
# Copy to the Android project
cp whats_in_my_thattu_v2.tflite \
  tensorImageInterpreter/src/main/ml/whats_in_my_thattu_v2.tflite
```

The Android app's `mlModelBinding = true` in `build.gradle.kts` will auto-generate
a `WhatsInMyThattuV2` class. Update `TensorImageInterpreter.kt` to use it:

```kotlin
// Change this line:
private val model: WhatsInMyThattu = WhatsInMyThattu.newInstance(context)

// To this:
private val model: WhatsInMyThattuV2 = WhatsInMyThattuV2.newInstance(context)
```

Then rebuild the app. The `TensorImageInterpreter` already has the correct
preprocessing (resize to 224x224 + normalize to [0,1]) to match this model.